# Visualización de grafos AnCora-ES-Semantic

Lee una muestra de los `.graphml` generados por `scripts/build_graphml_ancora.py`
y los dibuja usando la misma paleta y layout jerárquico de `tree_balance.py`.

- **Nodos**: el color codifica el tipo semántico (verbo / núcleo / predicativo / funcional).
- **Layout**: jerárquico, raíz arriba, agente elevado medio nivel.
- **Sin etiquetas de arista**: la información semántica está en el color y la posición.

In [10]:
!pip install -U networkx

ERROR: Could not install packages due to an OSError: HTTPSConnectionPool(host='files.pythonhosted.org', port=443): Max retries exceeded with url: /packages/9e/c9/b2622292ea83fbb4ec318f5b9ab867d0a28ab43c5717bb85b0a5f6b3b0a4/networkx-3.6.1-py3-none-any.whl.metadata (Caused by ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')))


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import random
import sys
from pathlib import Path

# Compatibilidad NumPy 2.x ↔ networkx <3.4: networkx 3.1 referencia np.float_ /
# np.int_, que NumPy 2.0 eliminó. Restauramos los alias antes de importar networkx.
import numpy as np
for _attr, _alias in [("float_", "float64"), ("int_", "int64")]:
    if not hasattr(np, _attr):
        setattr(np, _attr, getattr(np, _alias))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

# Importar paleta y layout desde el script hermano
sys.path.insert(0, str(Path.cwd() / "scripts"))
from tree_balance import (
    COLOR_VERBO, BORDE_VERBO, TEXTO_VERBO,
    COLOR_NUCLEO, BORDE_NUCLEO, TEXTO_NUCLEO,
    COLOR_PRED, BORDE_PRED, TEXTO_PRED,
    COLOR_FUNC, BORDE_FUNC, TEXTO_FUNC,
    calcular_layout, color_nodo,
)

CARPETA = Path("AnCora-ES-Semantic")
assert CARPETA.exists(), f"No existe la carpeta {CARPETA}"

In [2]:
def cargar(ruta):
    """Carga un .graphml y devuelve (G, tipos_nodo, verbo_raiz, aristas, phrase)."""
    G = nx.read_graphml(ruta)
    tipos_nodo = {n: data.get("tipo", "funcional") for n, data in G.nodes(data=True)}
    verbo_raiz = G.graph.get("root")
    aristas = [(u, v, "dep") for u, v in G.edges()]
    phrase = G.graph.get("phrase", "")
    return G, tipos_nodo, verbo_raiz, aristas, phrase


def dibujar(ax, G, tipos_nodo, verbo_raiz, aristas, titulo):
    """Dibuja un grafo sobre el axes dado. Versión inline de generar_grafo."""
    pos = calcular_layout(G, verbo_raiz, tipos_nodo, aristas)

    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    ax.set_axis_off()
    ax.set_xlim(min(xs) - 1.5, max(xs) + 1.5)
    ax.set_ylim(min(ys) - 1.2, max(ys) + 1.2)
    ax.set_title(titulo, fontsize=9, fontweight="bold", color="#333333", pad=8)

    NODE_H = 0.35
    CHAR_W = 0.13

    # Aristas (debajo)
    for u, v in G.edges():
        if u not in pos or v not in pos:
            continue
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        dx, dy = x2 - x1, y2 - y1
        dist = (dx ** 2 + dy ** 2) ** 0.5
        if dist == 0:
            continue
        retro = NODE_H / 2
        x2r = x2 - (dx / dist) * retro
        y2r = y2 - (dy / dist) * retro
        ax.annotate("", xy=(x2r, y2r), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="-|>", color="#999999",
                                    lw=0.8, mutation_scale=12))

    # Nodos
    for nodo, (x, y) in pos.items():
        tipo = tipos_nodo.get(nodo, "funcional")
        fill, borde, txt = color_nodo(tipo)
        w = max(0.8, len(str(nodo)) * CHAR_W)
        lw = 1.5 if tipo == "verbo" else (1.1 if tipo == "nucleo" else 0.7)
        box = mpatches.FancyBboxPatch(
            (x - w / 2, y - NODE_H / 2), w, NODE_H,
            boxstyle="round,pad=0.06",
            facecolor=fill, edgecolor=borde, linewidth=lw, zorder=3,
        )
        ax.add_patch(box)
        fs = 7 if len(str(nodo)) > 14 else (8 if len(str(nodo)) > 9 else 9)
        fw = "bold" if tipo in ("verbo", "nucleo") else "normal"
        ax.text(x, y, str(nodo), ha="center", va="center",
                fontsize=fs, fontweight=fw, color=txt, zorder=4)


def leyenda(fig):
    handles = [
        mpatches.Patch(facecolor=COLOR_VERBO, edgecolor=BORDE_VERBO, label="verbo"),
        mpatches.Patch(facecolor=COLOR_NUCLEO, edgecolor=BORDE_NUCLEO, label="núcleo de rol"),
        mpatches.Patch(facecolor=COLOR_PRED, edgecolor=BORDE_PRED, label="predicativo"),
        mpatches.Patch(facecolor=COLOR_FUNC, edgecolor=BORDE_FUNC, label="funcional"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=9,
               frameon=True, edgecolor="#cccccc", bbox_to_anchor=(0.5, -0.02))

## Muestra aleatoria de grafos de tamaño medio

Filtramos por número de nodos (8 a 18) para que sean legibles en la grilla.

In [3]:
TAMANO_MIN, TAMANO_MAX = 8, 100
N_MUESTRA = 6
SEMILLA = 42

rng = random.Random(SEMILLA)
todos = list(CARPETA.glob("*.graphml"))
print(f"Disponibles: {len(todos)}")

# Buscar archivos en el rango de tamaño (muestreo con rechazo para no leer 17k archivos)
muestra = todos[0:10]
intentos = 0

print(f"Seleccionados {len(muestra)} en {intentos} intentos")
for p in muestra:
    print(" -", p.name)

Disponibles: 17310
Seleccionados 10 en 0 intentos
 - CESS-CAST-A_15209_20001220_s00007.graphml
 - CESS-CAST-P_148_19991001_s00006.graphml
 - CESS-CAST-AA_2698_20000203_s00006.graphml
 - CESS-CAST-P_84_19990902_s00007.graphml
 - CESS-CAST-P_132_20010301_s00011.graphml
 - CESS-CAST-P_135_19991201_e_s00002.graphml
 - 3LB-CAST_d2-1_s00009.graphml
 - CESS-CAST-A_10268_20000413_s00012.graphml
 - CESS-CAST-P_10_20010402_b_s00005.graphml
 - CESS-CAST-A_16182_20000220_s00017.graphml


In [4]:
filas = (len(muestra) + 1) // 2
fig, axes = plt.subplots(filas, 2, figsize=(16, 5.5 * filas))
axes = axes.flatten() if filas > 1 else [axes] if not hasattr(axes, "__iter__") else axes

for ax, ruta in zip(axes, muestra):
    G, tipos, raiz, aristas, phrase = cargar(ruta)
    # Recortar la frase para el título
    titulo = phrase if len(phrase) <= 90 else phrase[:87] + "…"
    titulo = f"{ruta.stem}\n{titulo}"
    dibujar(ax, G, tipos, raiz, aristas, titulo)

for ax in axes[len(muestra):]:
    ax.set_axis_off()

leyenda(fig)
plt.tight_layout()
plt.subplots_adjust(bottom=0.05)
plt.show()

/var/folders/rn/46s8dp4j7zng96g_212pqjlh0000gp/T/ipykernel_64882/1333678437.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Visualizar un grafo específico por nombre

Cambia `NOMBRE` por el archivo que quieras inspeccionar en detalle.

In [5]:
NOMBRE = "3LB-CAST_104_c-1_s00001.graphml"

ruta = CARPETA / NOMBRE
G, tipos, raiz, aristas, phrase = cargar(ruta)

print(f"{NOMBRE}")
print(f"Frase: {phrase}")
print(f"Raíz : {raiz}")
print(f"Nodos: {G.number_of_nodes()}, aristas: {G.number_of_edges()}")

fig, ax = plt.subplots(figsize=(14, 9))
dibujar(ax, G, tipos, raiz, aristas, NOMBRE)
leyenda(fig)
plt.tight_layout()
plt.subplots_adjust(bottom=0.07)
plt.show()

3LB-CAST_104_c-1_s00001.graphml
Frase: Las reservas de oro y divisas de Rusia subieron 800 millones de dólares y el 26_de_mayo equivalían a 19.100 millones de dólares, informó hoy un comunicado del Banco_Central.
Raíz : informó
Nodos: 20, aristas: 19


/var/folders/rn/46s8dp4j7zng96g_212pqjlh0000gp/T/ipykernel_64882/2785457192.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Distribución de tamaños del corpus

Histograma rápido del número de nodos por grafo (lee los 17k archivos, tarda unos segundos).

In [6]:
import xml.etree.ElementTree as ET

# Más rápido que nx.read_graphml: contamos <node ...> directamente
NS = "{http://graphml.graphdrawing.org/xmlns}"

tamanos = []
for ruta in todos:
    try:
        tree = ET.parse(ruta)
        tamanos.append(sum(1 for _ in tree.getroot().iter(NS + "node")))
    except Exception:
        pass

import statistics
print(f"N      : {len(tamanos)}")
print(f"Min/Max: {min(tamanos)} / {max(tamanos)}")
print(f"Media  : {statistics.mean(tamanos):.1f}")
print(f"Mediana: {statistics.median(tamanos)}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(tamanos, bins=range(1, max(tamanos) + 2, 2), color="#9FE1CB", edgecolor="#0F6E56")
ax.set_xlabel("Número de nodos por grafo")
ax.set_ylabel("Cantidad de grafos")
ax.set_title("Distribución de tamaños — AnCora-ES-Semantic")
ax.set_xlim(0, 60)
plt.tight_layout()
plt.show()

N      : 17310
Min/Max: 1 / 75
Media  : 17.8
Mediana: 17.0


/var/folders/rn/46s8dp4j7zng96g_212pqjlh0000gp/T/ipykernel_64882/468161260.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
